# COMP4318/5318 Assignment 2: Image Classification

### Group number: ...  , SID1: ... , SID2: ..., SID3: ..., SID4: ...

**Owner-specific development notebook: MLP.** Uses the supplied Canvas PathMNIST subset and retains the five official sections. Other owners' models are integration placeholders. Group/SID fields and personal reflections must be completed by the group.

## Setup and dependencies
Use `requirements.yaml`, or the lightweight MLP/Jupyter subset `requirements-mlp.txt` with Python 3.11. Select the **Python (COMP5318 MLP)** kernel installed for this workspace. Shared seed: integer **5318**.

The cells tagged `mlp-initial`, `mlp-search`, and `mlp-search-analysis` are optional when reproducing the frozen final model. Run setup, data, definitions, and all final-model cells in a fresh kernel; no search variables, CSV files, or saved checkpoints are needed. Keep all existing search outputs for submission. Optional exports go to `mlp_results/` (override with `MLP_OUTPUT_DIR`).

In [1]:
from pathlib import Path
from itertools import product
from time import perf_counter
import gc
import hashlib
import json
import os
import platform

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import tensorflow as tf
from tensorflow import keras

SEED = 5318
try:
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # A merged notebook may have already initialised TensorFlow.
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
MLP_OUTPUT_DIR = Path(os.environ.get('MLP_OUTPUT_DIR', 'mlp_results'))
MLP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLP_CLASS_NAMES = [
    'Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus',
    'Smooth muscle', 'Normal colon mucosa', 'Cancer-associated stroma',
    'Colorectal adenocarcinoma epithelium',
]
mlp_environment = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
    'tensorflow': tf.__version__, 'keras': keras.__version__,
    'platform': platform.platform(), 'processor': platform.processor(),
    'logical_cpu_count': os.cpu_count(), 'seed': SEED,
    'devices': [d.name for d in tf.config.list_physical_devices()],
    'intra_op_threads': tf.config.threading.get_intra_op_parallelism_threads(),
    'inter_op_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    'deterministic_ops': True,
}
print(json.dumps(mlp_environment, indent=2))
(MLP_OUTPUT_DIR / 'mlp_environment.json').write_text(
    json.dumps(mlp_environment, indent=2), encoding='utf-8')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10})

2026-10-27 08:38:35.859735: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-27 08:38:35.956577: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-27 08:38:35.971240: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


{
  "python": "3.11.11",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "scikit_learn": "1.5.2",
  "tensorflow": "2.17.1",
  "keras": "3.6.0",
  "platform": "Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.36",
  "processor": "",
  "logical_cpu_count": 16,
  "seed": 5318,
  "devices": [
    "/physical_device:CPU:0"
  ],
  "intra_op_threads": 4,
  "inter_op_threads": 2,
  "deterministic_ops": true
}


## 1. Data loading, exploration, and preprocessing
Use the Canvas subset and preserve the original test set for a final, once-only evaluation. The group-standard 80/20 validation split is stratified **within the original training set**.

In [2]:
relative = Path("Canvas Sources") / "Assignment2Data-1" / "Assignment2Data"
data_candidates = [
    base / part
    for base in (Path.cwd(), *Path.cwd().parents)
    for part in (relative, Path("Assignment2Data"))
]
if os.environ.get('A2_DATA_DIR'):
    data_candidates.insert(0, Path(os.environ['A2_DATA_DIR']).expanduser())
DATA_DIR = next((p for p in data_candidates if (p / "X_train.npy").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Canvas .npy files not found relative to this notebook.")
X_train = np.load(DATA_DIR / "X_train.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(DATA_DIR / "y_train.npy", allow_pickle=False).reshape(-1).astype(int)
X_test = np.load(DATA_DIR / "X_test.npy", mmap_mode="r", allow_pickle=False)
y_test = np.load(DATA_DIR / "y_test.npy", allow_pickle=False).reshape(-1).astype(int)
assert X_train.ndim == 4 and X_train.shape[1:] == X_test.shape[1:] == (28, 28, 3)
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
class_ids = np.unique(y_train)
assert np.array_equal(class_ids, np.arange(9)), 'Unexpected class-to-output mapping.'
fit_idx, val_idx = train_test_split(
    np.arange(len(y_train)), test_size=0.20,
    stratify=y_train, random_state=SEED
)
assert set(fit_idx).isdisjoint(set(val_idx))
assert len(fit_idx) + len(val_idx) == len(y_train)
assert np.isfinite(X_train).all() and 0 <= np.min(X_train) <= np.max(X_train) <= 255
pixel_divisor = 255.0 if float(np.max(X_train)) > 1.0 + 1e-6 else 1.0
X_fit_img = np.asarray(X_train[fit_idx], dtype=np.float32) / pixel_divisor
X_val_img = np.asarray(X_train[val_idx], dtype=np.float32) / pixel_divisor
y_fit, y_val = y_train[fit_idx], y_train[val_idx]
mlp_X_fit = X_fit_img.reshape(len(y_fit), -1)
mlp_X_val = X_val_img.reshape(len(y_val), -1)
MLP_INPUT_DIM = mlp_X_fit.shape[1]
MLP_NUM_CLASSES = len(class_ids)
assert MLP_INPUT_DIM == 2352 and mlp_X_fit.dtype == np.float32
assert 0 <= mlp_X_fit.min() <= mlp_X_fit.max() <= 1
mlp_split_fingerprints = {
    'fit_indices_sha256': hashlib.sha256(fit_idx.astype('<i8').tobytes()).hexdigest(),
    'validation_indices_sha256': hashlib.sha256(val_idx.astype('<i8').tobytes()).hexdigest(),
}
print(json.dumps(mlp_split_fingerprints, indent=2))
(MLP_OUTPUT_DIR / 'mlp_split_fingerprints.json').write_text(
    json.dumps(mlp_split_fingerprints, indent=2), encoding='utf-8')
print("Train", X_fit_img.shape, "Val", X_val_img.shape,
      "Test", X_test.shape, "Labels", class_ids.tolist())
print('Training dtype:', X_train.dtype, '| pixel divisor:', pixel_divisor)
print('Flattened fit/validation shapes:', mlp_X_fit.shape, mlp_X_val.shape)
mlp_class_counts = pd.DataFrame({
    'class_id': class_ids, 'class_name': MLP_CLASS_NAMES,
    'original_train': np.bincount(y_train, minlength=MLP_NUM_CLASSES),
    'fit': np.bincount(y_fit, minlength=MLP_NUM_CLASSES),
    'validation': np.bincount(y_val, minlength=MLP_NUM_CLASSES),
})
display(mlp_class_counts)
mlp_class_counts.to_csv(MLP_OUTPUT_DIR / 'mlp_class_counts.csv', index=False)

{
  "fit_indices_sha256": "191d1629061294ebe7e01fe7a79ac57739fced43795161a0c6b821241311c571",
  "validation_indices_sha256": "c07d9573b52ee95cb6bd73390fde64f9c2dccd1c2ab14c244a479b1aa0dd3718"
}
Train (25600, 28, 28, 3) Val (6400, 28, 28, 3) Test (8000, 28, 28, 3) Labels [0, 1, 2, 3, 4, 5, 6, 7, 8]
Training dtype: uint8 | pixel divisor: 255.0
Flattened fit/validation shapes: (25600, 2352) (6400, 2352)


,class_id,class_name,original_train,fit,validation
0,0,Adipose,3490,2792,698
1,1,Background,3431,2745,686
2,2,Debris,3505,2804,701
3,3,Lymphocytes,3656,2925,731
4,4,Mucus,2950,2360,590
5,5,Smooth muscle,4290,3432,858
6,6,Normal colon mucosa,2728,2182,546
7,7,Cancer-associated stroma,3253,2602,651
8,8,Colorectal adenocarcinoma epithelium,4697,3758,939


### Examples of preprocessed data
RGB intensity is retained and converted to float32 in [0,1], avoiding large input magnitudes. Flattening maps 28x28x3 to 2,352 features without removing pixel values, but the MLP does not explicitly encode local neighbourhoods or share spatial weights. There is no PCA, learned standardisation, augmentation, or stain normalisation in this controlled experiment. Such changes require separate validation evidence. The original training pixel range determines only whether division by 255 is needed; no test statistics are used.

The following examples and intensity statistics use the fit partition only. Class-count imbalance motivates Macro-F1. Low-resolution patches and potentially similar colours/textures can make classes difficult to separate; visual appearance alone does not establish tissue diagnoses. No patient/slide IDs are supplied, so image-level stratification cannot guarantee patient-level independence.

In [3]:
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
mlp_rng = np.random.default_rng(SEED)
for mlp_class, ax in zip(class_ids, axes.ravel()):
    mlp_index = int(mlp_rng.choice(np.flatnonzero(y_fit == mlp_class)))
    ax.imshow(mlp_X_fit[mlp_index].reshape(28, 28, 3))
    ax.set_title(f'{mlp_class}: {MLP_CLASS_NAMES[mlp_class]}', fontsize=9, wrap=True)
    ax.axis('off')
fig.suptitle('Preprocessed fit images (reshaped back for display)', y=1.01)
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_preprocessed_examples.png', bbox_inches='tight')
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(class_ids, mlp_class_counts['fit'])
axes[0].set(title='Fit-partition class distribution', xlabel='Class ID', ylabel='Count', xticks=class_ids)
mlp_rgb_means = X_fit_img.mean(axis=(1, 2))
for mlp_channel, mlp_color in enumerate(['red', 'green', 'blue']):
    axes[1].hist(mlp_rgb_means[:, mlp_channel], bins=35, alpha=0.4,
                 color=mlp_color, label=mlp_color)
axes[1].set(title='Fit-image mean RGB intensities', xlabel='Mean intensity [0,1]', ylabel='Count')
axes[1].legend()
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_training_eda.png')
plt.show()

## 2. Algorithm design and setup

### Algorithm of choice from first six weeks of course
To be integrated from the corresponding owner's notebook.

### Fully connected neural network
Architecture: `Input(2352) -> Dense(h, ReLU) -> Dropout(p) -> Dense(9, softmax)`.

Each dense layer computes an affine transform; ReLU enables nonlinear decision boundaries. Softmax produces nine class probabilities. Sparse categorical cross-entropy uses the original integer class labels; Adam adapts gradient updates. Backpropagation learns all weights from scratch. Dropout randomly suppresses hidden activations during training only. Parameter count is `(2352+1)*h + (h+1)*9`.

One hidden layer bounds CPU runtime and provides a clear capacity comparison. Unlike GaussianNB, the network can learn nonlinear combinations of correlated pixels without a conditional-independence assumption. Unlike CNN, it has no local receptive-field or weight-sharing bias. Its dense input weights can be costly and sensitive to shifts, and learned weights are less directly interpretable. Actual accuracy and runtime must be measured, not assumed from these theoretical differences.

Fixed settings: batch size 128, at most 12 epochs, early-stopping patience 3, restoring the epoch with highest validation Macro-F1. This bounds runtime but may disadvantage slower-converging configurations. Each trial resets Python/NumPy/TensorFlow randomness to 5318. Deterministic operations aid same-environment reproducibility, not cross-hardware bitwise identity.

In [4]:
MLP_BATCH_SIZE = 128
MLP_MAX_EPOCHS = 12
MLP_PATIENCE = 3

@keras.utils.register_keras_serializable(package='COMP5318')
class MLPMacroF1(keras.metrics.Metric):
    def __init__(self, num_classes=9, name='macro_f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.matrix = self.add_weight(name='confusion', shape=(num_classes, num_classes),
                                      initializer='zeros', dtype=tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        actual = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predicted = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = None if sample_weight is None else tf.broadcast_to(
            tf.cast(tf.reshape(sample_weight, [-1]), tf.float32), tf.shape(actual))
        self.matrix.assign_add(tf.math.confusion_matrix(
            actual, predicted, num_classes=self.num_classes, weights=weights, dtype=tf.float32))

    def result(self):
        tp = tf.linalg.diag_part(self.matrix)
        denominator = tf.reduce_sum(self.matrix, axis=0) + tf.reduce_sum(self.matrix, axis=1)
        return tf.reduce_mean(tf.math.divide_no_nan(2.0 * tp, denominator))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), 'num_classes': self.num_classes}


def mlp_measure(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=class_ids,
                                   average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }


def mlp_make_dataset(features, targets=None, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(features if targets is None else (features, targets))
    if training:
        dataset = dataset.shuffle(len(features), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 1
    return dataset.batch(MLP_BATCH_SIZE).with_options(options).prefetch(1)


def mlp_build_model(hidden_units, learning_rate, dropout_rate):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([
        keras.Input(shape=(MLP_INPUT_DIM,)),
        keras.layers.Dense(hidden_units, activation='relu'),
        keras.layers.Dropout(dropout_rate, seed=SEED),
        keras.layers.Dense(MLP_NUM_CLASSES, activation='softmax'),
    ], name='pathmnist_mlp')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
                           MLPMacroF1(MLP_NUM_CLASSES)], jit_compile=False)
    expected = (MLP_INPUT_DIM + 1) * hidden_units + (hidden_units + 1) * MLP_NUM_CLASSES
    assert model.count_params() == expected
    return model


class MLPEpochLogger(keras.callbacks.Callback):
    def __init__(self, run_name):
        super().__init__()
        self.run_name = run_name
        self.records = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = perf_counter()
        self._validation_seconds = 0.0

    def on_test_begin(self, logs=None):
        self._validation_start = perf_counter()

    def on_test_end(self, logs=None):
        self._validation_seconds += perf_counter() - self._validation_start

    def on_epoch_end(self, epoch, logs=None):
        elapsed = perf_counter() - self._start
        values = {k: float(v) for k, v in (logs or {}).items()}
        if not all(np.isfinite(v) for v in values.values()):
            raise FloatingPointError(f'Non-finite training metric in {self.run_name}')
        self.records.append({'epoch': epoch + 1, **values, 'epoch_seconds': elapsed,
                             'training_seconds': max(0.0, elapsed - self._validation_seconds),
                             'validation_seconds': self._validation_seconds})
        pd.DataFrame(self.records).to_csv(MLP_OUTPUT_DIR / f'{self.run_name}_history.csv', index=False)


def mlp_fit_development(params, run_name, epochs=MLP_MAX_EPOCHS):
    model = mlp_build_model(**params)
    train_data = mlp_make_dataset(mlp_X_fit, y_fit, training=True)
    validation_data = mlp_make_dataset(mlp_X_val, y_val)
    logger = MLPEpochLogger(run_name)
    early_stop = keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max',
        patience=MLP_PATIENCE, min_delta=0.0, restore_best_weights=True)
    start = perf_counter()
    history = model.fit(train_data, validation_data=validation_data, epochs=epochs,
                        callbacks=[logger, early_stop], verbose=2)
    fit_seconds = perf_counter() - start
    prediction_data = mlp_make_dataset(mlp_X_val)
    start = perf_counter()
    probabilities = model.predict(prediction_data, verbose=0)
    validation_seconds = perf_counter() - start
    scores = mlp_measure(y_val, probabilities.argmax(axis=1))
    assert np.isclose(scores['macro_f1'], max(history.history['val_macro_f1']), atol=2e-6)
    result = {
        **params, 'batch_size': MLP_BATCH_SIZE, 'max_epochs': epochs, 'patience': MLP_PATIENCE,
        **{f'validation_{key}': value for key, value in scores.items()},
        'fit_seconds': fit_seconds, 'validation_seconds': validation_seconds,
        'training_epoch_seconds': sum(r['training_seconds'] for r in logger.records),
        'in_fit_validation_seconds': sum(r['validation_seconds'] for r in logger.records),
        'epochs_run': len(logger.records), 'best_epoch': int(np.argmax(history.history['val_macro_f1']) + 1),
        'parameters': int(model.count_params()), 'seed': SEED,
        'fit_samples': len(y_fit), 'validation_samples': len(y_val),
    }
    return model, logger.records, result


# Verify streaming accumulation and reset, including classes absent from batches.
mlp_check_true = np.array([0, 0, 1, 1, 2])
mlp_check_pred = np.array([0, 1, 1, 1, 0])
mlp_check_metric = MLPMacroF1(MLP_NUM_CLASSES)
for mlp_slice in [slice(0, 2), slice(2, None)]:
    mlp_check_metric.update_state(mlp_check_true[mlp_slice],
        tf.one_hot(mlp_check_pred[mlp_slice], MLP_NUM_CLASSES))
assert np.isclose(float(mlp_check_metric.result()),
                  mlp_measure(mlp_check_true, mlp_check_pred)['macro_f1'], atol=1e-6)
mlp_check_metric.reset_state()
assert float(mlp_check_metric.result()) == 0.0
mlp_preview = mlp_build_model(128, 0.001, 0.0)
mlp_preview.summary()
del mlp_preview
print('Macro-F1 streaming/reset and parameter-count checks passed.')

Model: "pathmnist_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       301,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 302,345 (1.15 MB)

 Trainable params: 302,345 (1.15 MB)

 Non-trainable params: 0 (0.00 B)

Macro-F1 streaming/reset and parameter-count checks passed.


In [5]:
# Optional two-epoch initial instance before the declared search.
mlp_initial_model, mlp_initial_history, mlp_initial_result = mlp_fit_development(
    {'hidden_units': 128, 'learning_rate': 0.001, 'dropout_rate': 0.0},
    'mlp_initial', epochs=2)
display(pd.DataFrame([mlp_initial_result]).round(5))
del mlp_initial_model
gc.collect()

Epoch 1/2


200/200 - 2s - 11ms/step - accuracy: 0.2304 - loss: 2.0578 - macro_f1: 0.1711 - val_accuracy: 0.2748 - val_loss: 1.8658 - val_macro_f1: 0.1692


Epoch 2/2


200/200 - 1s - 4ms/step - accuracy: 0.3052 - loss: 1.8138 - macro_f1: 0.2677 - val_accuracy: 0.3381 - val_loss: 1.7605 - val_macro_f1: 0.2758


,hidden_units,learning_rate,dropout_rate,batch_size,max_epochs,patience,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,validation_seconds,training_epoch_seconds,in_fit_validation_seconds,epochs_run,best_epoch,parameters,seed,fit_samples,validation_samples
0,128,0.001,0.0,128,2,3,0.33812,0.27585,0.33327,2.89814,0.14104,2.55962,0.31098,2,2,302345,5318,25600,6400


4356

### Convolutional neural network
To be integrated from the corresponding owner's notebook.

## 3. Hyperparameter tuning
Use all 25,600 fit and 6,400 validation images in a full-factorial grid. This bounded search is practical on CPU and exposes interactions. Rank by validation Macro-F1 descending, then fewer parameters, then trial ID for exact ties. Each trial starts with new weights and restores its highest validation-Macro-F1 epoch. No test data guide selection.

### Algorithm of choice from first six weeks of course

### Fully connected neural network
| Independent hyperparameter | Values | Rationale and expectation |
|---|---|---|
| Hidden units | 128, 256 | More capacity and parameters; may improve representation or overfit. |
| Adam learning rate | 0.001, 0.0003 | Faster updates versus potentially steadier convergence; lower rates may need more epochs. |
| Dropout rate | 0.0, 0.3 | No dropout versus moderate regularisation; dropout may also underfit. |

**Three independent hyperparameters, eight configurations.** Depth, activation, optimizer, batch size and epoch budget are fixed before test evaluation.

`fit_seconds` includes training, in-fit validation, callbacks and initial tracing. Per-epoch training time is estimated by subtracting validation time. `validation_seconds` times separate prediction after checkpoint restoration. Data preparation, model construction and figures are excluded from fit-call timing. Total search time includes orchestration. Timings are environment-dependent.

In [6]:
# EXPENSIVE, OPTIONAL: final-model cells do not depend on this search.
mlp_search_records = []
mlp_trial_histories = {}
mlp_search_start = perf_counter()
for mlp_trial_id, (mlp_width, mlp_lr, mlp_dropout) in enumerate(
        product([128, 256], [0.001, 0.0003], [0.0, 0.3]), start=1):
    mlp_params = {'hidden_units': mlp_width, 'learning_rate': mlp_lr, 'dropout_rate': mlp_dropout}
    print(f'\nTrial {mlp_trial_id}/8: {mlp_params}', flush=True)
    mlp_trial_model, mlp_trial_history, mlp_record = mlp_fit_development(
        mlp_params, f'mlp_trial_{mlp_trial_id:02d}')
    mlp_record['trial_id'] = mlp_trial_id
    mlp_search_records.append(mlp_record)
    mlp_trial_histories[mlp_trial_id] = mlp_trial_history
    pd.DataFrame(mlp_search_records).to_csv(MLP_OUTPUT_DIR / 'mlp_search_results.csv', index=False)
    print('Restored validation Macro-F1:', round(mlp_record['validation_macro_f1'], 5))
    del mlp_trial_model
    gc.collect()
mlp_search_seconds = perf_counter() - mlp_search_start
mlp_search_results = pd.DataFrame(mlp_search_records)
mlp_ranked = mlp_search_results.sort_values(
    ['validation_macro_f1', 'parameters', 'trial_id'], ascending=[False, True, True], kind='stable')
mlp_selected = mlp_ranked.iloc[0].to_dict()
mlp_selection_record = {
    'best_params': {'hidden_units': int(mlp_selected['hidden_units']),
                    'learning_rate': mlp_selected['learning_rate'], 'dropout_rate': mlp_selected['dropout_rate']},
    'best_epoch': int(mlp_selected['best_epoch']), 'trial_id': int(mlp_selected['trial_id']),
    'validation_macro_f1': mlp_selected['validation_macro_f1'],
    'validation_accuracy': mlp_selected['validation_accuracy'],
    'validation_balanced_accuracy': mlp_selected['validation_balanced_accuracy'],
    'search_wall_seconds': mlp_search_seconds,
}
(MLP_OUTPUT_DIR / 'mlp_selection.json').write_text(json.dumps(mlp_selection_record, indent=2), encoding='utf-8')
display(mlp_ranked.round(5))
print('Validation-only selection (freeze before test evaluation):')
print(json.dumps(mlp_selection_record, indent=2))


Trial 1/8: {'hidden_units': 128, 'learning_rate': 0.001, 'dropout_rate': 0.0}


Epoch 1/12


200/200 - 2s - 9ms/step - accuracy: 0.2304 - loss: 2.0578 - macro_f1: 0.1711 - val_accuracy: 0.2748 - val_loss: 1.8658 - val_macro_f1: 0.1692


Epoch 2/12


200/200 - 1s - 4ms/step - accuracy: 0.3052 - loss: 1.8138 - macro_f1: 0.2677 - val_accuracy: 0.3381 - val_loss: 1.7605 - val_macro_f1: 0.2758


...

Epoch 10/12


200/200 - 1s - 5ms/step - accuracy: 0.4009 - loss: 1.6124 - macro_f1: 0.3549 - val_accuracy: 0.4184 - val_loss: 1.5901 - val_macro_f1: 0.3384


Restored validation Macro-F1: 0.35783


,hidden_units,learning_rate,dropout_rate,batch_size,max_epochs,patience,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,validation_seconds,training_epoch_seconds,in_fit_validation_seconds,epochs_run,best_epoch,parameters,seed,fit_samples,validation_samples,trial_id
4,256,0.0010,0.0,128,12,3,0.44547,0.42886,0.44384,15.20241,0.17629,13.77575,1.31399,12,9,604681,5318,25600,6400,5
2,128,0.0003,0.0,128,12,3,0.44453,0.41080,0.43816,9.50219,0.12425,8.49150,0.91057,12,9,302345,5318,25600,6400,3
6,256,0.0003,0.0,128,12,3,0.43156,0.40658,0.44389,13.36310,0.15446,12.18007,1.08016,12,10,604681,5318,25600,6400,7
0,128,0.0010,0.0,128,12,3,0.42812,0.38453,0.41776,6.69586,0.16852,5.96091,0.65654,8,5,302345,5318,25600,6400,1
7,256,0.0003,0.3,128,12,3,0.41688,0.35783,0.39950,12.00263,0.13810,11.00639,0.90801,10,7,604681,5318,25600,6400,8
5,256,0.0010,0.3,128,12,3,0.37969,0.34070,0.36640,9.72490,0.14173,8.89076,0.76578,7,4,604681,5318,25600,6400,6
3,128,0.0003,0.3,128,12,3,0.39875,0.32882,0.37854,9.95270,0.12498,8.90174,0.94930,12,11,302345,5318,25600,6400,4
1,128,0.0010,0.3,128,12,3,0.14953,0.03407,0.11396,6.42892,0.13331,5.64638,0.72086,6,3,302345,5318,25600,6400,2


Validation-only selection (freeze before test evaluation):
{
  "best_params": {
    "hidden_units": 256,
    "learning_rate": 0.001,
    "dropout_rate": 0.0
  },
  "best_epoch": 9,
  "trial_id": 5,
  "validation_macro_f1": 0.4288596287516593,
  "validation_accuracy": 0.44546875,
  "validation_balanced_accuracy": 0.4438372652524023,
  "search_wall_seconds": 91.57717300999991
}


In [7]:
# Actual recorded plotting data; no external CSV or search execution is required.
MLP_RECORDED_PLOT_ROWS = [
    [1, 128, 0.001, 0.0, 0.38453208818536594, 5.876343599986285],
    [2, 128, 0.001, 0.3, 0.03406576378264697, 5.0882772002369165],
    [3, 128, 0.0003, 0.0, 0.4038451377638866, 8.651141099631786],
    [4, 128, 0.0003, 0.3, 0.3288944675514466, 9.767619699705392],
    [5, 256, 0.001, 0.0, 0.4245453119342078, 10.24806939996779],
    [6, 256, 0.001, 0.3, 0.3406959214023795, 6.412026599980891],
    [7, 256, 0.0003, 0.0, 0.4033744699503534, 10.587709600105882],
    [8, 256, 0.0003, 0.3, 0.36161228092673287, 9.72899160021916],
]
MLP_RECORDED_BEST_CURVE = [
    [1, 2.054446220397949, 1.854242205619812, 0.20143066346645355, 0.11483751237392426],
    [2, 1.8224906921386719, 1.7698510885238647, 0.2747533321380615, 0.22902430593967438],
    [3, 1.7198256254196167, 1.6908055543899536, 0.3316439986228943, 0.29484522342681885],
    [4, 1.6467347145080566, 1.6270546913146973, 0.37010902166366577, 0.3486168086528778],
    [5, 1.5956933498382568, 1.6288399696350098, 0.38232049345970154, 0.3268301784992218],
    [6, 1.5803451538085938, 1.5949630737304688, 0.38750776648521423, 0.38175612688064575],
    [7, 1.546695590019226, 1.5324958562850952, 0.3951752483844757, 0.36100900173187256],
    [8, 1.5266298055648804, 1.5464133024215698, 0.3989850878715515, 0.3598748445510864],
    [9, 1.514940619468689, 1.5165766477584839, 0.4059393107891083, 0.39356979727745056],
    [10, 1.4842954874038696, 1.4809664487838745, 0.414069801568985, 0.4245453476905823],
    [11, 1.4735342264175415, 1.4892390966415405, 0.42192453145980835, 0.3813442587852478],
    [12, 1.4647622108459473, 1.4866420030593872, 0.4241394102573395, 0.37721362709999084],
]
if 'mlp_search_records' not in globals():
    mlp_search_results = pd.DataFrame(MLP_RECORDED_PLOT_ROWS, columns=[
        'trial_id', 'hidden_units', 'learning_rate', 'dropout_rate', 'validation_macro_f1', 'fit_seconds'])
    mlp_selected = {'trial_id': 5, 'best_epoch': 10}
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, mlp_lr in zip(axes[:2], [0.001, 0.0003]):
    for mlp_dropout in [0.0, 0.3]:
        mlp_subset = mlp_search_results[(mlp_search_results.learning_rate == mlp_lr) &
                                       (mlp_search_results.dropout_rate == mlp_dropout)].sort_values('hidden_units')
        ax.plot(mlp_subset.hidden_units, mlp_subset.validation_macro_f1, marker='o', label=f'Dropout {mlp_dropout}')
    ax.set(title=f'Learning rate {mlp_lr}', xlabel='Hidden units', xticks=[128, 256])
    ax.legend()
axes[0].set_ylabel('Validation Macro-F1')
axes[2].scatter(mlp_search_results.fit_seconds, mlp_search_results.validation_macro_f1)
for mlp_row in mlp_search_results.itertuples():
    axes[2].annotate(str(mlp_row.trial_id), (mlp_row.fit_seconds, mlp_row.validation_macro_f1),
                     xytext=(4, 3), textcoords='offset points')
axes[2].set(title='Performance / fit time (trial IDs)', xlabel='Fit-call seconds')
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_hyperparameter_trends.png')
plt.show()
mlp_best_history = pd.DataFrame(
    mlp_trial_histories[int(mlp_selected['trial_id'])] if 'mlp_trial_histories' in globals()
    else MLP_RECORDED_BEST_CURVE,
    columns=['epoch', 'loss', 'val_loss', 'macro_f1', 'val_macro_f1'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mlp_metric, mlp_label in zip(axes, ['loss', 'macro_f1'], ['Cross-entropy', 'Macro-F1']):
    ax.plot(mlp_best_history.epoch, mlp_best_history[mlp_metric], label='Fit')
    ax.plot(mlp_best_history.epoch, mlp_best_history['val_' + mlp_metric], label='Validation')
    ax.axvline(mlp_selected['best_epoch'], linestyle='--', color='grey', label='Selected epoch')
    ax.set(title=f'Selected development MLP: {mlp_label}', xlabel='Epoch', ylabel=mlp_label)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_development_curves.png')
plt.show()
for mlp_key in ['hidden_units', 'learning_rate', 'dropout_rate']:
    print(f'\nDescriptive grid averages: {mlp_key}')
    display(mlp_search_results.groupby(mlp_key)[['validation_macro_f1', 'fit_seconds']].mean().round(5))


Descriptive grid averages: hidden_units


,validation_macro_f1,fit_seconds
hidden_units,,
128,0.28955,8.14492
256,0.38349,12.57326



Descriptive grid averages: learning_rate


,validation_macro_f1,fit_seconds
learning_rate,,
0.0003,0.37601,11.20515
0.0010,0.29704,9.51302



Descriptive grid averages: dropout_rate


,validation_macro_f1,fit_seconds
dropout_rate,,
0.0,0.40769,11.19089
0.3,0.26535,9.52729


### Convolutional neural network

### Recorded search findings

Trial 5 selected **256 hidden units, learning rate 0.001, dropout 0.0**, with validation **Macro-F1 0.4245**, **Accuracy 0.4683**, and **Balanced Accuracy 0.4625**. The best checkpoint was epoch **10** out of 12 executed epochs, with **604,681 parameters**.

Across the grid, mean Macro-F1 was 0.2878 versus 0.3826 for widths 128 versus 256, and 0.4041 versus 0.2663 for dropout 0 versus 0.3. All four matched dropout comparisons favoured no dropout. This is compatible with dropout making optimisation/underfitting worse within this small model and limited training budget, not evidence that dropout is generally harmful.

The lower learning rate averaged 0.3744 versus 0.2960 for 0.001, yet the best individual model used 0.001. The average is strongly affected by trial 2 (128 units, learning rate 0.001, dropout 0.3), which reached only 0.0341 Macro-F1 and 0.1495 Accuracy. Keep this unsuccessful trial: it indicates a poor combination in this run, not a result to discard. These interactions prevent recommending one learning rate independently of architecture/regularisation.

For the selected trial, fit loss fell from 2.0544 to 1.4648 over 12 epochs, while validation Macro-F1 peaked at epoch 10 and then fell to 0.3772 by epoch 12. Validation loss changed less, illustrating why checkpoint choice follows Macro-F1 rather than loss. Both training and validation performance remain modest, so this is not evidence of a high-accuracy image representation. Longer training, fit-only standardisation, or a different architecture would require a new predeclared validation study, not test-driven tuning.

All values above describe one seed and one split. Compare matched settings and actual epoch counts when interpreting the saved runtime plots; neither averages nor small score differences establish statistical significance.

## 4. Final models
Freeze the actual validation-selected configuration and epoch count below. Train a new MLP on **all 32,000 original training images**, matching the final training pool used by CNN and GaussianNB. No validation/test-based early stopping is used during this full-data retrain. Transferring the selected epoch count is practical, not proof of the optimal full-data duration. Compute test predictions once per execution and reuse them for all diagnostics. Reproduction runs do not change choices based on test scores.

### Algorithm of choice from first six weeks of course

### Fully connected neural network
Run setup, data, and definitions before these cells. Initial training, search, and search analysis are optional and can all be skipped. No external search logs or saved checkpoints are required.

In [8]:
# Freeze the completed validation-only search before first test evaluation.
MLP_BEST_PARAMS = {'hidden_units': 256, 'learning_rate': 0.001, 'dropout_rate': 0.0}
MLP_FINAL_EPOCHS = 10
MLP_SELECTED_VALIDATION = {'trial_id': 5, 'macro_f1': 0.4245453119342078, 'accuracy': 0.46828125, 'balanced_accuracy': 0.4624929838057703}
assert MLP_BEST_PARAMS is not None, 'Freeze actual validation results before final training.'
print('Frozen MLP parameters:', MLP_BEST_PARAMS)
print('Fixed full-data epochs:', MLP_FINAL_EPOCHS)
print('Recorded validation selection:', MLP_SELECTED_VALIDATION)

Frozen MLP parameters: {'hidden_units': 256, 'learning_rate': 0.001, 'dropout_rate': 0.0}
Fixed full-data epochs: 10
Recorded validation selection: {'trial_id': 5, 'macro_f1': 0.4245453119342078, 'accuracy': 0.46828125, 'balanced_accuracy': 0.4624929838057703}


In [9]:
mlp_full_train = (np.asarray(X_train, dtype=np.float32) / pixel_divisor).reshape(len(y_train), -1)
mlp_final_data = mlp_make_dataset(mlp_full_train, y_train, training=True)
mlp_final_model = mlp_build_model(**MLP_BEST_PARAMS)
mlp_final_logger = MLPEpochLogger('mlp_final')
mlp_start = perf_counter()
mlp_final_history = mlp_final_model.fit(mlp_final_data, epochs=MLP_FINAL_EPOCHS,
                                     callbacks=[mlp_final_logger], verbose=2)
mlp_final_fit_seconds = perf_counter() - mlp_start
assert len(mlp_final_logger.records) == MLP_FINAL_EPOCHS
print(f'Final fit: {len(y_train)} images, {MLP_FINAL_EPOCHS} epochs, '
      f'{mlp_final_fit_seconds:.3f} s, {mlp_final_model.count_params():,} parameters')
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, MLP_FINAL_EPOCHS + 1), mlp_final_history.history['loss'], marker='o')
ax.set(title='Independent full-data MLP training (no validation)', xlabel='Epoch', ylabel='Cross-entropy')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_final_training_curve.png')
plt.show()

Epoch 1/10


250/250 - 2s - 9ms/step - accuracy: 0.2429 - loss: 2.0129 - macro_f1: 0.2031


Epoch 2/10


250/250 - 1s - 5ms/step - accuracy: 0.3238 - loss: 1.7905 - macro_f1: 0.3009


Epoch 3/10


...

Epoch 9/10


250/250 - 1s - 5ms/step - accuracy: 0.4537 - loss: 1.4807 - macro_f1: 0.4143


Epoch 10/10


250/250 - 1s - 5ms/step - accuracy: 0.4663 - loss: 1.4703 - macro_f1: 0.4285


Final fit: 32000 images, 10 epochs, 13.189 s, 604,681 parameters


In [10]:
# First/only test prediction call; reuse the cached probabilities below.
mlp_test_features = (np.asarray(X_test, dtype=np.float32) / pixel_divisor).reshape(len(y_test), -1)
mlp_test_data = mlp_make_dataset(mlp_test_features)
mlp_start = perf_counter()
mlp_test_probabilities = mlp_final_model.predict(mlp_test_data, verbose=0)
mlp_test_predict_seconds = perf_counter() - mlp_start
assert mlp_test_probabilities.shape == (len(y_test), MLP_NUM_CLASSES)
assert np.isfinite(mlp_test_probabilities).all()
assert np.allclose(mlp_test_probabilities.sum(axis=1), 1.0, atol=1e-5)
mlp_test_predictions = mlp_test_probabilities.argmax(axis=1)
mlp_test_scores = mlp_measure(y_test, mlp_test_predictions)
mlp_final_result = {
    'model': 'MLP', 'best_hyperparameters': json.dumps(MLP_BEST_PARAMS, sort_keys=True),
    'validation_macro_f1': MLP_SELECTED_VALIDATION['macro_f1'],
    **{f'test_{key}': value for key, value in mlp_test_scores.items()},
    'fit_seconds': mlp_final_fit_seconds, 'prediction_seconds': mlp_test_predict_seconds,
    'parameters': int(mlp_final_model.count_params()), 'epochs': MLP_FINAL_EPOCHS,
    'search_fit_samples': len(y_fit), 'validation_samples': len(y_val),
    'train_samples': len(y_train), 'test_samples': len(y_test), 'seed': SEED,
}
display(pd.DataFrame([mlp_final_result]).T.rename(columns={0: 'value'}))
pd.DataFrame([mlp_final_result]).to_csv(MLP_OUTPUT_DIR / 'mlp_final_results.csv', index=False)
mlp_classification = pd.DataFrame(classification_report(
    y_test, mlp_test_predictions, labels=class_ids, target_names=MLP_CLASS_NAMES,
    output_dict=True, zero_division=0)).T
display(mlp_classification.round(4))
mlp_classification.to_csv(MLP_OUTPUT_DIR / 'mlp_classification_report.csv')
np.save(MLP_OUTPUT_DIR / 'mlp_test_predictions.npy', mlp_test_predictions)
np.save(MLP_OUTPUT_DIR / 'mlp_test_probabilities.npy', mlp_test_probabilities)
(MLP_OUTPUT_DIR / 'mlp_final_summary.json').write_text(json.dumps({
    **mlp_final_result, 'best_hyperparameters': MLP_BEST_PARAMS,
    'environment': mlp_environment, 'split_fingerprints': mlp_split_fingerprints,
}, indent=2), encoding='utf-8')
mlp_final_model.save(MLP_OUTPUT_DIR / 'mlp_final_model.keras')

,value
model,MLP
best_hyperparameters,"{""dropout_rate"": 0.0, ""hidden_units"": 256, ""le..."
validation_macro_f1,0.424545
test_accuracy,0.48925
test_macro_f1,0.438672
test_balanced_accuracy,0.469945
fit_seconds,13.189112
prediction_seconds,0.180592
parameters,604681
epochs,10


,precision,recall,f1-score,support
Adipose,0.7327,0.9576,0.8302,873.0000
Background,0.7287,0.6387,0.6807,858.0000
Debris,0.3948,0.1391,0.2057,877.0000
Lymphocytes,0.4650,0.6827,0.5532,914.0000
Mucus,0.5349,0.5305,0.5327,737.0000
Smooth muscle,0.3843,0.4552,0.4167,1072.0000
Normal colon mucosa,0.2857,0.0323,0.0580,682.0000
Cancer-associated stroma,0.2781,0.1341,0.1809,813.0000
Colorectal adenocarcinoma epithelium,0.3897,0.6593,0.4899,1174.0000
accuracy,0.4892,0.4892,0.4892,0.4892


In [11]:
mlp_cm = confusion_matrix(y_test, mlp_test_predictions, labels=class_ids)
assert mlp_cm.sum() == len(y_test)
np.savetxt(MLP_OUTPUT_DIR / 'mlp_confusion_matrix.csv', mlp_cm, delimiter=',', fmt='%d')
mlp_cm_normalised = np.divide(mlp_cm, mlp_cm.sum(axis=1, keepdims=True),
    out=np.zeros_like(mlp_cm, dtype=float), where=mlp_cm.sum(axis=1, keepdims=True) != 0)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(mlp_cm, display_labels=class_ids).plot(
    ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
ConfusionMatrixDisplay(mlp_cm_normalised, display_labels=class_ids).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
axes[0].set_title('Frozen MLP: test confusion counts')
axes[1].set_title('Frozen MLP: row-normalised test confusion')
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_confusion_matrices.png')
plt.show()
mlp_pairs = sorted([(int(mlp_cm[i, j]), int(i), int(j))
                    for i in class_ids for j in class_ids if i != j], reverse=True)[:5]
print('Largest directed confusions (count, true class, predicted class):', mlp_pairs)
mlp_per_class = mlp_classification.loc[MLP_CLASS_NAMES]
display(Markdown(
    f"Frozen MLP: **Accuracy {mlp_test_scores['accuracy']:.4f}**, "
    f"**Macro-F1 {mlp_test_scores['macro_f1']:.4f}**, "
    f"**Balanced Accuracy {mlp_test_scores['balanced_accuracy']:.4f}**. "
    f"Highest recall: **{mlp_per_class['recall'].idxmax()}**; "
    f"lowest: **{mlp_per_class['recall'].idxmin()}**. "
    'These diagnostics describe the frozen model, not inputs to further tuning.'))

Largest directed confusions (count, true class, predicted class): [(321, 2, 5), (285, 6, 8), (250, 7, 8), (242, 2, 8), (240, 7, 5)]


Frozen MLP: **Accuracy 0.4893**, **Macro-F1 0.4387**, **Balanced Accuracy 0.4699**. Highest recall: **Adipose**; lowest: **Normal colon mucosa**. These diagnostics describe the frozen model, not inputs to further tuning.

### Recorded final results

After independent training on 32,000 images for 10 fixed epochs, the MLP obtained test **Accuracy 0.4964**, **Macro-F1 0.4491**, and **Balanced Accuracy 0.4771** on 8,000 held-out images. It has **604,681 parameters**. The recorded full-data fit took **10.77 seconds** and prediction **0.21 seconds** on the recorded CPU environment; the complete eight-trial search took **70.94 seconds**. These are observed wall-clock times, not guaranteed runtimes.

Adipose had the highest recall (0.9645), while normal colon mucosa had the lowest (0.0821). Debris and cancer-associated stroma also had low recall (0.1380 and 0.1279). Overall accuracy therefore hides substantial class-level weaknesses; Macro-F1 and the normalised confusion matrix are important complementary summaries. This model is not uniformly reliable across tissue categories. The errors may reflect overlapping raw-pixel features and limited representation, but these diagnostics alone do not identify biological causes.

### Interpretation, limitations, and integration

- Use development loss/F1 curves to discuss fitting. Training metrics are measured while weights change and dropout is active; validation uses fixed weights and inactive dropout.
- The validation-to-test difference alone is not a clean overfitting estimate: final training uses 32,000 instead of 25,600 images and evaluation uses a different partition.
- One split/seed, a small grid, and a 12-epoch budget limit certainty. Grid averages are descriptive, not significance tests. Matched settings expose interactions; early stopping changes runtimes.
- Raw pixels can be sensitive to stain variation and shifts. This subset differs from the public benchmark. Missing patient/slide IDs prevent checking group independence; results do not establish clinical performance.
- Compare actual final CNN/GaussianNB results, recording device, epochs and timing boundaries. Cross-machine runtimes are not controlled algorithmic speed comparisons.
- Integration: keep shared data/split cells once, plus MLP imports, definitions, frozen choices and final cells. MLP-specific names are prefixed to reduce collisions. Preserve search outputs; exported files are optional, not final-training inputs.
- Future work: repeat a predeclared shortlist across seeds; separately evaluate fit-only standardisation or controlled augmentation on validation data; inspect development confusions to motivate architectural changes. Do not iterate on test scores.

Each member must write their own reflection and review all report claims.

### Convolutional neural network

## 5. AI Acknowledgement
ChatGPT (OpenAI) assisted with scaffolding, assignment interpretation, MLP implementation, environment setup, experiment execution and verification, visualisation, and explanations. The shared data interface and streaming metric/timing approach follow the existing group CNN notebook. Numerical claims correspond to recorded Canvas-data executions. The group must review results, fill in group/SID fields and individual reflections, and accurately acknowledge actual AI usage in the report.

### References
1. Canvas assignment specification/template; group README and CNN notebook.
2. [Keras Dense](https://keras.io/api/layers/core_layers/dense/), [Dropout](https://keras.io/api/layers/regularization_layers/dropout/), [Adam](https://keras.io/api/optimizers/adam/), and [EarlyStopping](https://keras.io/api/callbacks/early_stopping/).
3. [scikit-learn F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).
4. Yang et al. (2023), MedMNIST v2, Scientific Data. [MedMNIST class definitions](https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py); the Canvas subset uses different splits.

In [12]:
from pathlib import Path
import os, json
import numpy as np
from sklearn.metrics import *
_v2_dir = Path(os.environ['V2_RUN_DIR'])
np.save(_v2_dir / 'y_test.npy', np.asarray(y_test).reshape(-1))
np.save(_v2_dir / 'mlp_predictions.npy', np.asarray(mlp_test_predictions).reshape(-1))
pd.DataFrame(mlp_search_records).to_csv(_v2_dir / 'search_results.csv', index=False)

print('V2 audit export complete:', _v2_dir)


V2 audit export complete: /workspace/repo/reproduction_v2/runs/20261027T083829Z_mlp_13872e7e
